## WP012 — Market inefficiency: is there any bettable edge?

See `README.md` for the question, the data, and — importantly — the **three pre-declared primary tests (P1–P3) and what each outcome means**. Those were fixed before this notebook was run; every other table here is exploratory.

No sampling in this notebook: it runs in about a minute. Helpers live in `football_model.evaluation.market` (unit-tested, including simulated markets with a known edge).

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from football_model.evaluation import market as mk

REPO = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP001 = REPO / 'work_products' / 'wp001_walkforward_cv_baseline'
WP003 = REPO / 'work_products' / 'wp003_bookmaker_benchmark'

N_BOOT = 5000
TAU_PRIMARY = 0.02            # fixed in the README, not tuned
TAUS = [-np.inf, 0.0, 0.01, 0.02, 0.03, 0.05]

def show(df, digits=4):
    print(df.round(digits).to_string(index=False))

odds = pd.read_pickle(WP003 / 'odds_raw.pkl').dropna(subset=['Date', 'FTR']).reset_index(drop=True)
print(len(odds), 'matches,', odds['Date'].min().date(), '->', odds['Date'].max().date())
for label, prefix in [('Pinnacle pre-closing', 'PS'), ('Pinnacle closing', 'PSC'), ('Bet365 closing', 'B365C'),
                      ('Average closing', 'AvgC'), ('Max closing', 'MaxC')]:
    have = ~np.isnan(mk.odds_matrix(odds, prefix)).any(axis=1)
    print(f'  {label:<22} complete on {have.sum()} / {len(odds)} ({have.mean():.1%})')

2280 matches, 2020-09-12 -> 2026-05-24
  Pinnacle pre-closing   complete on 2110 / 2280 (92.5%)
  Pinnacle closing       complete on 2110 / 2280 (92.5%)
  Bet365 closing         complete on 2280 / 2280 (100.0%)
  Average closing        complete on 2280 / 2280 (100.0%)
  Max closing            complete on 2280 / 2280 (100.0%)


## 1. Market-only tests (model-free, all matches)

### 1a. Is Pinnacle's closing line calibrated?

De-vigged closing probabilities pooled over the three outcomes of every match, binned. A favourite-longshot bias would show as hit rate above the probability in the top bins and below it in the bottom ones. Wilson 95% intervals; outcomes within a match are not independent, so treat the intervals as slightly optimistic.

In [2]:
psc = mk.odds_matrix(odds, 'PSC')
ok = ~np.isnan(psc).any(axis=1)
p_pin_all = mk.devig(psc[ok])
y_all = mk.outcome_onehot(odds.loc[ok, 'FTR'])

show(mk.calibration_table(p_pin_all, y_all, [0, .1, .2, .3, .4, .5, .6, .7, .8, 1.0]))
print()
print('by outcome (mean fair probability vs realised frequency):')
print(pd.DataFrame({'mean_p': p_pin_all.mean(0), 'rate': y_all.mean(0)}, index=mk.OUTCOMES).round(4))

       bin    n  mean_p   rate     lo     hi
  [0, 0.1]  257  0.0725 0.0584 0.0357 0.0941
[0.1, 0.2] 1087  0.1576 0.1582 0.1377 0.1811
[0.2, 0.3] 2306  0.2557 0.2515 0.2342 0.2696
[0.3, 0.4]  868  0.3420 0.3399 0.3091 0.3720
[0.4, 0.5]  653  0.4467 0.4518 0.4140 0.4901
[0.5, 0.6]  505  0.5518 0.5604 0.5168 0.6031
[0.6, 0.7]  332  0.6461 0.6536 0.6009 0.7028
[0.7, 0.8]  226  0.7437 0.7434 0.6827 0.7959
  [0.8, 1]   96  0.8393 0.8854 0.8064 0.9348

by outcome (mean fair probability vs realised frequency):
   mean_p    rate
H  0.4362  0.4341
D  0.2394  0.2313
A  0.3243  0.3346


### 1b. Blind ROI by odds band (favourite-longshot check)

Flat-stake every outcome whose closing odds fall in each band, at that book's own price. With a fair market every band would sit at minus the margin; a favourite-longshot bias shows as long-odds bands losing more than short-odds bands. `all` is the control: it should be about minus each book's overround.

In [3]:
BAND_EDGES = [1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 1e9]
for name, (_, closing) in mk.BOOKS.items():
    o = mk.odds_matrix(odds, closing)
    ok = ~np.isnan(o).any(axis=1)
    y = mk.outcome_onehot(odds.loc[ok, 'FTR'])
    t = mk.odds_band_table(o[ok], y, BAND_EDGES, n_boot=N_BOOT)
    allrow = mk.odds_band_table(o[ok], y, [0, 1e9], n_boot=N_BOOT).assign(band='all')
    print(f'--- {name} closing ({ok.sum()} matches) ---')
    show(pd.concat([t, allrow]))
    print()

--- pinnacle closing (2110 matches) ---
      band  n_bets  implied  hit_rate     roi      lo      hi
  [1, 1.5)     463   0.7652    0.7473 -0.0246 -0.0784  0.0273
  [1.5, 2)     760   0.5805    0.5776 -0.0065 -0.0679  0.0539
    [2, 3)    1118   0.4138    0.4168  0.0084 -0.0522  0.0706
    [3, 5)    2728   0.2675    0.2540 -0.0475 -0.1006  0.0075
    [5, 8)     844   0.1672    0.1552 -0.0711 -0.2144  0.0743
[8, 1e+09)     417   0.0901    0.0839 -0.1173 -0.3946  0.1854
       all    6330   0.3422    0.3333 -0.0388 -0.0630 -0.0139



--- bet365 closing (2280 matches) ---
      band  n_bets  implied  hit_rate     roi      lo      hi
  [1, 1.5)     512   0.7700    0.7324 -0.0508 -0.0998 -0.0015
  [1.5, 2)     868   0.5850    0.5714 -0.0245 -0.0808  0.0335
    [2, 3)    1214   0.4189    0.3962 -0.0531 -0.1079  0.0016
    [3, 5)    2970   0.2716    0.2552 -0.0607 -0.1123 -0.0087
    [5, 8)     880   0.1707    0.1602 -0.0528 -0.1881  0.0879
[8, 1e+09)     396   0.0949    0.0732 -0.2664 -0.5199  0.0067
       all    6840   0.3516    0.3333 -0.0649 -0.0858 -0.0440



--- average closing (2280 matches) ---
      band  n_bets  implied  hit_rate     roi      lo      hi
  [1, 1.5)     504   0.7643    0.7321 -0.0442 -0.0947  0.0055
  [1.5, 2)     843   0.5819    0.5706 -0.0222 -0.0789  0.0354
    [2, 3)    1255   0.4142    0.4008 -0.0342 -0.0878  0.0207
    [3, 5)    2990   0.2690    0.2548 -0.0534 -0.1056 -0.0017
    [5, 8)     865   0.1663    0.1584 -0.0428 -0.1812  0.1035
[8, 1e+09)     383   0.0897    0.0731 -0.2252 -0.4981  0.0627
       all    6840   0.3477    0.3333 -0.0536 -0.0750 -0.0322



--- max closing (2280 matches) ---
      band  n_bets  implied  hit_rate     roi      lo     hi
  [1, 1.5)     432   0.7557    0.7407 -0.0224 -0.0766 0.0329
  [1.5, 2)     801   0.5807    0.5980  0.0292 -0.0301 0.0894
    [2, 3)    1198   0.4154    0.4224  0.0206 -0.0419 0.0825
    [3, 5)    2928   0.2642    0.2599 -0.0162 -0.0693 0.0378
    [5, 8)     947   0.1682    0.1774  0.0605 -0.0796 0.2050
[8, 1e+09)     534   0.0896    0.0861 -0.0859 -0.3380 0.1763
       all    6840   0.3318    0.3333  0.0003 -0.0236 0.0242



### 1c. Soft-vs-sharp value bets

Treat Pinnacle's de-vigged closing probability as the truth and bet a **soft** book's closing price whenever that price is worth more than `tau` in expected value: `edge = soft_odds × p_pinnacle − 1 > tau`. Settled at the soft book's price, with the real outcome. `-inf` bets everything (control: minus the book's margin). `claimed_edge` is what Pinnacle's fair prices said the chosen bets were worth; compare to realised `roi`.

Two things flatter this test and are not real: no book restricts you here, and `Max` is the best price across many books, so it is line-shopping's upper bound, not one account.

In [4]:
psc = mk.odds_matrix(odds, 'PSC')
soft_results = {}
for name in ['bet365', 'average', 'max']:
    soft = mk.odds_matrix(odds, mk.BOOKS[name][1])
    ok = ~np.isnan(psc).any(axis=1) & ~np.isnan(soft).any(axis=1)
    p_fair = mk.devig(psc[ok])
    y = mk.outcome_onehot(odds.loc[ok, 'FTR'])
    soft_results[name] = (ok, p_fair, soft[ok], y)
    print(f'--- {name} closing vs Pinnacle fair ({ok.sum()} matches) ---')
    show(mk.roi_table(p_fair, soft[ok], y, TAUS, n_boot=N_BOOT))
    print()

--- bet365 closing vs Pinnacle fair (2110 matches) ---


 tau  n_bets  n_matches     roi      lo      hi  claimed_edge  hit_rate
-inf    6330       2110 -0.0699 -0.0923 -0.0468       -0.0570    0.3333
0.00     202        191 -0.1168 -0.3867  0.1840        0.0250    0.1832
0.01     123        117 -0.0615 -0.4348  0.3673        0.0384    0.1707
0.02      75         71 -0.2287 -0.6974  0.3429        0.0539    0.1067
0.03      49         46 -0.1306 -0.7328  0.6611        0.0694    0.1224
0.05      29         28 -0.0828 -0.8668  0.9227        0.0904    0.1379

--- average closing vs Pinnacle fair (2110 matches) ---


 tau  n_bets  n_matches     roi      lo      hi  claimed_edge  hit_rate
-inf    6330       2110 -0.0571 -0.0803 -0.0334       -0.0439    0.3333
0.00     116        112 -0.1648 -0.5059  0.2203        0.0173    0.1983
0.01      62         61  0.0011 -0.5186  0.6079        0.0294    0.2258
0.02      35         34 -0.3517 -0.8049  0.2039        0.0410    0.1714
0.03      15         15 -0.5293 -1.0000  0.2526        0.0639    0.1333
0.05       9          9 -0.7533 -1.0000 -0.1675        0.0801    0.1111

--- max closing vs Pinnacle fair (2110 matches) ---


 tau  n_bets  n_matches     roi      lo     hi  claimed_edge  hit_rate
-inf    6330       2110 -0.0025 -0.0286 0.0240        0.0144    0.3333
0.00    3527       1949 -0.0179 -0.0711 0.0343        0.0383    0.2900
0.01    2617       1731 -0.0071 -0.0777 0.0634        0.0500    0.2702
0.02    1971       1487 -0.0094 -0.0982 0.0773        0.0616    0.2501
0.03    1529       1233 -0.0106 -0.1187 0.0976        0.0723    0.2341
0.05     905        790 -0.0898 -0.2333 0.0528        0.0953    0.1989



## 2. Does the model add information beyond Pinnacle?

WP001's baseline predictions (35 walk-forward windows, 401 held-out EPL matches) joined to the odds file. The join **raises** if any stored score disagrees with the odds file's full-time score. Rows without Pinnacle pre-closing and closing odds are dropped, leaving the same ~361 matches as WP003.

In [5]:
with open(WP001 / 'cv_shared_data.pkl', 'rb') as f:
    shared = pickle.load(f)
df_cv, windows = shared['df_cv'], shared['windows']
with open(WP001 / 'cv_checkpoint.pkl', 'rb') as f:
    ckpt = pickle.load(f)

fixtures = mk.model_fixtures(df_cv, windows, ckpt)
need = [f'PS{o}' for o in mk.OUTCOMES] + [f'PSC{o}' for o in mk.OUTCOMES]
J = mk.join_odds(fixtures, odds, required_cols=need)
print(f'{len(fixtures)} model fixtures -> {len(J)} with Pinnacle pre-closing and closing odds; scores all agree with odds file')

p_mod = J[['p_home_model', 'p_draw_model', 'p_away_model']].to_numpy()
p_close = mk.devig(mk.odds_matrix(J, 'PSC'))
p_open = mk.devig(mk.odds_matrix(J, 'PS'))
open_odds = mk.odds_matrix(J, 'PS')
yJ = mk.outcome_onehot(J['FTR'])
y_idx = yJ.argmax(axis=1)
first_half, second_half = mk.half_masks(J['date'])
print('halves:', first_half.sum(), '/', second_half.sum())

401 model fixtures -> 361 with Pinnacle pre-closing and closing odds; scores all agree with odds file
halves: 176 / 185


### 2a. Blend sweep (RPS)

`p = (1 − w) × Pinnacle_close + w × model`, paired against pure Pinnacle (`w = 0`). If the model carried information the market lacks, a small `w` would *reduce* RPS below Pinnacle's. Negative `diff` = the blend beats Pinnacle. Choosing the best `w` here would be in-sample; this is a diagnostic, not P1.

In [6]:
rps_pin = mk.rps(p_close, yJ)
rows = []
for w in [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 1.0]:
    r = mk.rps(mk.blend(p_close, p_mod, w), yJ)
    m, lo, hi = mk.bootstrap_ci(r - rps_pin, N_BOOT)
    rows.append({'w_model': w, 'rps': r.mean(), 'diff_vs_pinnacle': m, 'lo': lo, 'hi': hi})
show(pd.DataFrame(rows))

 w_model    rps  diff_vs_pinnacle     lo     hi
    0.00 0.1786            0.0000 0.0000 0.0000
    0.05 0.1790            0.0004 0.0001 0.0007
    0.10 0.1795            0.0009 0.0003 0.0014
    0.20 0.1805            0.0019 0.0007 0.0030
    0.30 0.1816            0.0030 0.0012 0.0047
    0.50 0.1841            0.0055 0.0026 0.0084
    1.00 0.1925            0.0139 0.0080 0.0198


### 2b. Leave-one-window-out logit — **P1**

Multinomial logistic regression on log-odds features, each window predicted from a fit on the other 34 (no match sees its own outcome). `market_only` refits a calibration on Pinnacle's log-odds alone; `market_plus_model` adds the model's log-odds. The per-match log-loss difference isolates what the model adds *after* the market's own information is recalibrated — the refit cost is paid by both sides. Negative = the model helps.

In [7]:
X_market = mk.logodds_features(p_close)
X_both = np.hstack([X_market, mk.logodds_features(p_mod)])
groups = J['window'].to_numpy()

loss_market = mk.leave_group_out_logloss(X_market, y_idx, groups)
loss_both = mk.leave_group_out_logloss(X_both, y_idx, groups)
d_p1 = loss_both - loss_market

raw = lambda p: -np.log(p[np.arange(len(p)), y_idx])
print(f'raw Pinnacle log loss      {raw(p_close).mean():.4f}')
print(f'raw model log loss         {raw(p_mod).mean():.4f}')
print(f'market_only (LOWO refit)   {loss_market.mean():.4f}')
print(f'market_plus_model (LOWO)   {loss_both.mean():.4f}')
m, lo, hi = mk.bootstrap_ci(d_p1, N_BOOT)
print(f'\nP1  paired difference (negative = model helps): {m:+.5f}  95% CI [{lo:+.5f}, {hi:+.5f}]')
print(f'    first half {d_p1[first_half].mean():+.5f}   second half {d_p1[second_half].mean():+.5f}')

raw Pinnacle log loss      0.9134
raw model log loss         0.9600
market_only (LOWO refit)   0.9180
market_plus_model (LOWO)   0.9250

P1  paired difference (negative = model helps): +0.00700  95% CI [-0.00400, +0.01829]
    first half +0.00750   second half +0.00652


## 3. Opening vs closing, and closing line value

### 3a. How good is the opening line, and where does the model sit relative to both?

Pinnacle's pre-closing price is what you could have bet earlier in the week. The market usually gets *more* accurate towards the close, so beating the opening line without beating the close is the realistic version of an edge (and is exactly what CLV measures below).

In [8]:
rps_open, rps_mod = mk.rps(p_open, yJ), mk.rps(p_mod, yJ)
rows = []
for label, d in [('Pinnacle open − Pinnacle close', rps_open - rps_pin),
                 ('model − Pinnacle open', rps_mod - rps_open),
                 ('model − Pinnacle close', rps_mod - rps_pin)]:
    m, lo, hi = mk.bootstrap_ci(d, N_BOOT)
    rows.append({'paired RPS difference': label, 'mean': m, 'lo': lo, 'hi': hi})
print(f'RPS  open {rps_open.mean():.4f}   close {rps_pin.mean():.4f}   model {rps_mod.mean():.4f}')
show(pd.DataFrame(rows))

RPS  open 0.1808   close 0.1786   model 0.1925
         paired RPS difference   mean     lo     hi
Pinnacle open − Pinnacle close 0.0023 0.0001 0.0044
         model − Pinnacle open 0.0117 0.0063 0.0172
        model − Pinnacle close 0.0139 0.0080 0.0198


### 3b. Does the model's disagreement with the opening line predict where the line moves?

Per outcome, `x = p_model − p_open` and `y = p_close − p_open`; slope through the origin, clustered by match. Slope 1 = the model fully anticipates the market's move, 0 = its disagreements are noise relative to the eventual close, negative = the market moves *against* the model. Exploratory.

In [9]:
x = p_mod - p_open
y = p_close - p_open
slope, lo, hi = mk.ratio_bootstrap((x * y).sum(axis=1), (x * x).sum(axis=1), N_BOOT)
print(f'slope of (close − open) on (model − open): {slope:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]')

slope of (close − open) on (model − open): -0.013  95% CI [-0.049, +0.024]


### 3c. CLV of model-selected bets at Pinnacle's opening price — **P2**

Bet every outcome where the model's expected value at Pinnacle's *opening* odds exceeds `tau` (`open_odds × p_model − 1 > tau`). **CLV** = mean `open_odds × p_close_fair − 1` over those bets: how much better than the eventual sharp close the price you took was. It needs far fewer bets than ROI to show a real edge, which is why professional bettors track it. `roi` is the realised return of the same bets. The `-inf` row bets everything and is the control (about minus Pinnacle's margin).

In [10]:
rows = []
for tau in TAUS:
    mask = mk.edge(p_mod, open_odds) > tau
    res = mk.clv_table(open_odds, p_close, mask, yJ, n_boot=N_BOOT)
    rows.append({'tau': tau, **res})
show(pd.DataFrame(rows))

 tau  n_bets     clv  clv_lo  clv_hi     roi  roi_lo  roi_hi
-inf    1083 -0.0291 -0.0327 -0.0252 -0.0875 -0.1346 -0.0376
0.00     453 -0.0289 -0.0391 -0.0185 -0.2081 -0.3560 -0.0540
0.01     432 -0.0291 -0.0395 -0.0186 -0.2075 -0.3584 -0.0442
0.02     416 -0.0301 -0.0409 -0.0191 -0.2336 -0.3878 -0.0670
0.03     390 -0.0325 -0.0436 -0.0209 -0.2976 -0.4533 -0.1276
0.05     366 -0.0344 -0.0459 -0.0223 -0.2998 -0.4634 -0.1199


## Primary tests — verdicts

The three tests fixed in the README, evaluated exactly as declared: 95% CI entirely on the favourable side of zero **and** both halves of the data (split at the median match date) on the favourable side.

In [11]:
def verdict(est, lo, hi, h1, h2, favourable):
    sign = 1 if favourable == 'positive' else -1
    ci_ok = (lo > 0) if sign > 0 else (hi < 0)
    halves_ok = (h1 * sign > 0) and (h2 * sign > 0)
    return 'HIT' if ci_ok and halves_ok else 'no hit'

rows = []

# P1 -- model adds information beyond the market (negative log-loss difference is good)
m, lo, hi = mk.bootstrap_ci(d_p1, N_BOOT)
h1, h2 = d_p1[first_half].mean(), d_p1[second_half].mean()
rows.append({'test': 'P1 log-loss, market+model − market-only', 'n': len(d_p1), 'estimate': m, 'lo': lo, 'hi': hi,
             'half_1': h1, 'half_2': h2, 'result': verdict(m, lo, hi, h1, h2, 'negative')})

# P2 -- CLV of model-selected bets at Pinnacle opening price
mask = mk.edge(p_mod, open_odds) > TAU_PRIMARY
full = mk.clv_table(open_odds, p_close, mask, n_boot=N_BOOT)
halves = [mk.clv_table(open_odds[h], p_close[h], mask[h], n_boot=N_BOOT)['clv'] for h in (first_half, second_half)]
rows.append({'test': f'P2 CLV per bet, tau={TAU_PRIMARY}', 'n': full['n_bets'], 'estimate': full['clv'], 'lo': full['clv_lo'],
             'hi': full['clv_hi'], 'half_1': halves[0], 'half_2': halves[1],
             'result': verdict(full['clv'], full['clv_lo'], full['clv_hi'], *halves, 'positive')})

# P3 -- Bet365 closing vs Pinnacle fair
ok, p_fair, soft, y = soft_results['bet365']
dates = odds.loc[ok, 'Date']
f1, f2 = mk.half_masks(dates)
t_full = mk.roi_table(p_fair, soft, y, [TAU_PRIMARY], n_boot=N_BOOT).iloc[0]
h_roi = [mk.roi_table(p_fair[h], soft[h], y[h], [TAU_PRIMARY], n_boot=N_BOOT).iloc[0]['roi'] for h in (f1, f2)]
rows.append({'test': f'P3 Bet365 ROI vs Pinnacle fair, tau={TAU_PRIMARY}', 'n': int(t_full['n_bets']), 'estimate': t_full['roi'],
             'lo': t_full['lo'], 'hi': t_full['hi'], 'half_1': h_roi[0], 'half_2': h_roi[1],
             'result': verdict(t_full['roi'], t_full['lo'], t_full['hi'], *h_roi, 'positive')})

summary = pd.DataFrame(rows)
show(summary, digits=5)

                                    test   n  estimate       lo       hi   half_1   half_2 result
 P1 log-loss, market+model − market-only 361   0.00700 -0.00400  0.01829  0.00750  0.00652 no hit
                P2 CLV per bet, tau=0.02 416  -0.03006 -0.04087 -0.01910 -0.01902 -0.04198 no hit
P3 Bet365 ROI vs Pinnacle fair, tau=0.02  75  -0.22867 -0.69738  0.34295  0.41098 -1.00000 no hit
